<a href="https://colab.research.google.com/github/mahadikprasad15/ARENA/blob/main/Probes_generalization_Offline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install transformer_lens

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import re
import random
import transformer_lens
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import plotly.express as px
import pandas as pd

In [ ]:
def has_list(text):
    """
    Detects if text contains a structured list.

    Rules:
    1. Must have markers (bullets or numbers)
    2. Must have newlines between items
    3. Must have at least 2 items
    """

    numbered_pattern = r'\n\s*\d+[\.)]\s+'
    bullet_pattern = r'\n\s*[-*•]\s+'

    numbered_matches = len(re.findall(numbered_pattern, text))
    bullet_matches = len(re.findall(bullet_pattern, text))


    if numbered_matches >= 2 or bullet_matches >= 2:
        return True

    return False

In [ ]:
print(has_list('''This is a normal sentence \n
1. This is somewhat list
2. This is getting there
3. This is definitely there'''))


print(has_list('''This is a normal sentence \n
* This is somewhat list
* This is getting there
* This is definitely there'''))

In [ ]:
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)


tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()


def has_list(text):

    numbered_pattern = r'(^|\n)\s*\d+[\.)]\s+'
    bullet_pattern   = r'(^|\n)\s*[-*•]\s+'

    numbered_matches = len(re.findall(numbered_pattern, text))
    bullet_matches   = len(re.findall(bullet_pattern, text))

    return (numbered_matches >= 2) or (bullet_matches >= 2)


def generate_text(prompt, model, tokenizer, max_new_tokens=80):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


def generate_list_training_data(n_samples=100, model=model, tokenizer=tokenizer):
    assert n_samples % 2 == 0

    data = []
    texts = []


    prompt_for_list = "Give a numbered list of 5 ideas about productivity.\n1."
    prompt_against_list = "Explain productivity in one paragraph. No bullets, no numbering."

    for _ in range(n_samples // 2):
        texts.append(generate_text(prompt_for_list, model, tokenizer))

    for _ in range(n_samples // 2):
        texts.append(generate_text(prompt_against_list, model, tokenizer))

    random.shuffle(texts)
    labels = [has_list(t) for t in texts]

    data = [{"text": texts[i], "has_list": labels[i]} for i in range(len(texts))]
    return data


In [ ]:
model_hooked = transformer_lens.HookedTransformer.from_pretrained('gpt2-small', device = device)

In [ ]:
def extract_activations(texts, model, tokenizer, layer_idx=-1):

    if layer_idx < 0:
      layer_idx = model.cfg.n_layers + layer_idx
    _, cache = model.run_with_cache(texts)
    activations = cache[f'blocks.{layer_idx}.hook_resid_post']
    mean_activations = activations.mean(dim = 1)

    return mean_activations.to(device)


In [ ]:
class Probe(nn.Module):
  def __init__(self, input_dim, output_dim):
    super().__init__()

    self.input_dim = input_dim
    self.output_dim = output_dim
    self.ln = nn.Linear(self.input_dim, self.output_dim)

  def forward(self, x):
    return self.ln(x)

In [ ]:
criterion = nn.CrossEntropyLoss()
num_epochs = 10

data = generate_list_training_data(n_samples = 10, model = model, tokenizer = tokenizer)
texts = [data[i]['text'] for i in range(len(data))]
labels = [data[i]['has_list'] for i in range(len(data))]
labels_tensor = torch.tensor(labels, dtype=torch.long).to(device)

probes_all = {layer: [] for layer in range(model_hooked.cfg.n_layers)}

In [ ]:
for layer in tqdm(range(model_hooked.cfg.n_layers)):
    probe = Probe(model_hooked.cfg.d_model, 2).to(device)
    activations_layer = extract_activations(texts, model_hooked, tokenizer, layer_idx=layer)
    optimizer = optim.SGD(probe.parameters(), lr=0.01)
    total_loss = []

    for epoch in range(num_epochs):
        optimizer.zero_grad()
        output = probe(activations_layer)
        loss = criterion(output, labels_tensor)

        total_loss.append(loss.item())

        loss.backward()
        optimizer.step()

    probes_all[layer].append(probe)
    probes_all[layer].append(total_loss)

In [ ]:
loss_data = torch.stack([torch.tensor(probes_all[i][-1]) for i in range(len(probes_all))])


In [ ]:
losses_list = loss_data.flatten().tolist()
epochs = list(range(num_epochs)) * model_hooked.cfg.n_layers
layers = [layer for layer in range(model_hooked.cfg.n_layers) for _ in range(num_epochs)]

df_loss = pd.DataFrame({
    'Epoch': epochs,
    'Loss': losses_list,
    'Layer': layers
})

fig = px.line(df_loss, x='Epoch', y='Loss', color='Layer', title='Loss over Epochs for Each Layer')
fig.show()